# LCEL Chains & Output Parsers

LCEL = LangChain Expression Language
It uses the pipe operator (|) to connect components:

prompt | llm | output_parser

This is called a chain — data flows left to right through each step.
Each component takes input, transforms it, passes to next.

# Setup


In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# Simplest Chain

In [3]:
prompt = ChatPromptTemplate.from_messages(([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
]))

# StrOutputParser extracts just the text from AIMessage
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"input": "What is Python?"})
print(type(result))
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
**Python** is a high‑level, general‑purpose programming language that emphasizes readability, simplicity, and rapid development. It was created by Guido van Rossum and first released in 1991. Here are the key points that define Python:

| Feature | What it means |
|---------|---------------|
| **Readable syntax** | Uses indentation to delimit blocks, making code look like plain English. |
| **Interpreted** | Code runs line‑by‑line in a Python interpreter, which speeds up testing and debugging. |
| **Dynamic typing** | Variables don’t need explicit type declarations; types are inferred at runtime. |
| **Extensive standard library** | “Batteries included” – modules for file I/O, networking, web services, data manipulation, and more. |
| **Cross‑platform** | Runs on Windows, macOS, Linux, and many other operating systems. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, and functional programming styles. |
| **Large ecosyst

## What happens inside the pipe

Input dict
    │

prompt.invoke({"input": "..."})  → ChatPromptValue
    │

llm.invoke(ChatPromptValue)      → AIMessage
    │

StrOutputParser().invoke(AIMessage) → plain string

Each | connects one output to the next input automatically.


# Chaining two LLM calls


## First chain — generates a topic summary


In [4]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following topic in 2 sentences."),
    ("human", "{topic}")
])

## Second chain — generates quiz question from summary

In [5]:
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create one quiz question based on this summary."),
    ("human", "{summary}")
])

In [6]:
summary_chain = summary_prompt | llm | StrOutputParser()
quiz_chain = quiz_prompt | llm | StrOutputParser()

In [7]:
summary = summary_chain.invoke({"topic": "Agentic AI"})
print("Summary:", summary)

quiz = quiz_chain.invoke({"summary": summary})
print("Quiz Question:", quiz)

Summary: Agentic AI refers to artificial intelligence systems endowed with a degree of autonomy and self‑directed behavior, enabling them to set goals, make decisions, and act independently within their environment. Unlike purely reactive or task‑specific models, agentic AI can adapt its strategies, pursue objectives, and learn from experience, thereby exhibiting a form of artificial agency.
Quiz Question: **Quiz Question**

Which of the following best captures the defining characteristic of *agentic AI*?

A. It performs tasks only when explicitly instructed by a human operator.  
B. It reacts to inputs without any internal goal-setting or learning.  
C. It possesses a degree of autonomy, can set its own goals, and adapts its strategies based on experience.  
D. It relies solely on pre‑programmed rules and never changes its behavior.

**Correct Answer:** C

**Explanation:**  
Agentic AI is distinguished by its ability to act independently—setting goals, making decisions, and learning f

# JSON Output Parser

In [8]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extractor. Always respond with valid JSON only. No explanation, no markdown, just JSON"""),
    ("human", "Extract name, age and city from: {text}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({
    "text": "My name is Muhammad Usman Khan, I am 19 years old and live in Karachi"
})

print(type(result))
print(result)
print(result["name"])
print(result["city"])

<class 'dict'>
{'name': 'Muhammad Usman Khan', 'age': 19, 'city': 'Karachi'}
Muhammad Usman Khan
Karachi


# Structured Output with Pydantic

In [9]:
from pydantic import BaseModel, Field
from typing import List

class PersonalInfo(BaseModel):
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age of the person")
    city: str = Field(description="City where the person lives")
    skills: List[str] = Field(description="List of skills mentioned")

structured_llm = llm.with_structured_output(PersonalInfo)

structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured information from the text."),
    ("human", "{text}")
])

structured_chain = structured_prompt | structured_llm

result = structured_chain.invoke({
    "text": "Muhammad Usman Khan is a 19 year old developer from Karachi who knows Python, LangChain and FastAPI."
})

print(type(result))
print(result.name)
print(result.age)
print(result.skills)

<class '__main__.PersonalInfo'>
Muhammad Usman Khan
19
['Python', 'LangChain', 'FastAPI']
